<a href="https://colab.research.google.com/github/Brittany1216/ThinkPythonAssignments/blob/main/Week15/Week15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import csv
import os
from datetime import datetime, timedelta

DATA_FILE = "garden_data.csv"

def add_plant():
    name = input("Plant Name: ").strip()
    if not name:
        print("❌ Plant name cannot be empty.")
        return

    p_dt = get_valid_date("Planting Date (YYYY-MM-DD): ")
    h_days = get_valid_int("Days to Maturity: ")
    w_freq = get_valid_int("Watering Frequency (days): ")

    with open(DATA_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([name, p_dt.strftime("%Y-%m-%d"), h_days, w_freq, p_dt.strftime("%Y-%m-%d")])
    print(f"✅ {name} added successfully.")

def check_harvest_forecast():
    print("\n--- 🍎 Harvest Forecast ---")
    today = datetime.now()
    with open(DATA_FILE, "r") as f:
        for row in csv.reader(f):
            name, p_date, h_days, _, _ = row
            harvest_dt = datetime.strptime(p_date, "%Y-%m-%d") + timedelta(days=int(h_days))

            if harvest_dt <= today:
                print(f"READY NOW: {name} was due on {harvest_dt.strftime('%Y-%m-%d')}!")
            else:
                days_left = (harvest_dt - today).days
                print(f"{name}: Expected harvest in {days_left} days.")

def water_alerts():
    print("\n--- 💧 Urgent Watering List ---")
    today = datetime.now()
    found_thirsty = False

    with open(DATA_FILE, "r") as f:
        for row in csv.reader(f):
            name, _, _, w_freq, last_w = row
            next_water = datetime.strptime(last_w, "%Y-%m-%d") + timedelta(days=int(w_freq))

            if today >= next_water:
                print(f"⚠️ {name} needs water! (Due: {next_water.strftime('%Y-%m-%d')})")
                found_thirsty = True

    if not found_thirsty:
        print("Plants are looking hydrated! No watering needed today.")

while True:
    print("\n--- Garden Tool Menu ---")
    choice = input("1. Add Plant\n2. Harvest Forecast\n3. Water Alerts\n4. Exit\nChoice: ")

    if choice == "1": add_plant()
    elif choice == "2": check_harvest_forecast()
    elif choice == "3": water_alerts()
    elif choice == "4": break

def save_plant(name, plant_date, days_to_harvest, water_freq):
    with open(DATA_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([name, plant_date, days_to_harvest, water_freq, plant_date])

def get_garden_status():
    today = datetime.now()
    print(f"\n--- Garden Status for {today.strftime('%Y-%m-%d')} ---")

    try:
        with open(DATA_FILE, "r") as f:
            reader = csv.reader(f)
            for row in reader:
                name, p_date, harvest_days, water_freq, last_watered = row

                p_dt = datetime.strptime(p_date, "%Y-%m-%d")
                harvest_dt = p_dt + timedelta(days=int(harvest_days))
                days_left = (harvest_dt - today).days

                lw_dt = datetime.strptime(last_watered, "%Y-%m-%d")
                next_water = lw_dt + timedelta(days=int(water_freq))
                needs_water = "💧 NEEDS WATER" if today >= next_water else "OK"

                print(f"[{name}] Harvest in: {max(0, days_left)} days | Status: {needs_water}")
    except FileNotFoundError:
        print("No garden data found yet!")

choice = input("1. Add Plant\n2. View Status\nChoice: ")

if choice == "1":
    name = input("Plant Name: ")
    p_date = input("Plant Date (YYYY-MM-DD): ")
    h_days = input("Days to Harvest: ")
    w_freq = input("Watering Frequency (days): ")
    save_plant(name, p_date, h_days, w_freq)
    print("Plant saved to log!")
elif choice == "2":
    get_garden_status()

def get_valid_date(prompt):
    while True:
        date_str = input(prompt)
        try:
            return datetime.strptime(date_str, "%Y-%m-%d")
        except ValueError:
            print("❌ Invalid format. Please use YYYY-MM-DD (e.g., 2023-05-20).")

def get_valid_int(prompt):
    while True:
        num_str = input(prompt)
        if num_str.isdigit() and int(num_str) >= 0:
            return int(num_str)
        print("❌ Invalid input. Please enter a positive number.")

def check_garden(mode):
    if not os.path.exists(DATA_FILE):
        print("📭 No garden data found. Add a plant first!")
        return

    today = datetime.now()
    found = False

    print(f"\n--- {'Harvest Forecast' if mode == 'h' else 'Water Alerts'} ---")

    with open(DATA_FILE, "r") as f:
        reader = csv.reader(f)
        for row in reader:
            try:
                name, p_str, h_days, w_freq, last_w_str = row

                if mode == 'h':
                    target_dt = datetime.strptime(p_str, "%Y-%m-%d") + timedelta(days=int(h_days))
                    diff = (target_dt - today).days
                    status = "READY!" if diff <= 0 else f"in {diff} days"
                    print(f"• {name}: {status}")

                elif mode == 'w':
                    next_w = datetime.strptime(last_w_str, "%Y-%m-%d") + timedelta(days=int(w_freq))
                    if today >= next_w:
                        print(f"⚠️ {name} needs water! (Due: {next_w.strftime('%Y-%m-%d')})")
                        found = True
            except (ValueError, IndexError):
                continue

    if mode == 'w' and not found:
        print("✨ All plants are hydrated!")

while True:
    print("\n--- Bulletproof Garden Tool ---")
    choice = input("1. Add Plant\n2. Harvest Forecast\n3. Water Alerts\n4. Exit\nChoice: ")

    if choice == "1": add_plant()
    elif choice == "2": check_garden('h')
    elif choice == "3": check_garden('w')
    elif choice == "4": break
    else: print("❌ Invalid menu choice.")